In [ ]:
# Cell 9: SegFormer-B3 Model (FIXED - interpolate dinamis)
import torch
from transformers import SegformerForSemanticSegmentation

class SegFormerB3Flood(nn.Module):
    def __init__(self, num_classes=10, pretrained=True):
        super().__init__()
        model_name = "nvidia/mit-b3"
        
        if pretrained:
            self.model = SegformerForSemanticSegmentation.from_pretrained(
                model_name,
                num_labels=num_classes,
                ignore_mismatched_sizes=True,
                reshape_last_stage=True,
            )
        else:
            from transformers import SegformerConfig
            config = SegformerConfig.from_pretrained(model_name)
            config.num_labels = num_classes
            config.reshape_last_stage = True
            self.model = SegformerForSemanticSegmentation(config)
    
    def forward(self, pixel_values):
        outputs = self.model(pixel_values=pixel_values)
        logits = outputs.logits  # (B, num_classes, H/4, W/4)
        
        # FIXED: Dynamic size based on input
        logits = F.interpolate(
            logits, 
            size=pixel_values.shape[-2:],  # (H, W) dari input
            mode='bilinear', 
            align_corners=False
        )
        return logits

def init_model(device='cuda'):
    model = SegFormerB3Flood(num_classes=NUM_CLASSES, pretrained=True)
    return model.to(device)

print("SegFormer-B3 model ready.")

In [ ]:
# Cell 10: Training Configuration & Metrics
from torch.optim import AdamW
import math

class FloodSegMetrics:
    def __init__(self, num_classes, ignore_classes={2, 6}):
        self.num_classes = num_classes
        self.ignore_classes = ignore_classes
        self.reset()
    
    def reset(self):
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes), dtype=np.int64)
    
    def update(self, preds, targets):
        preds = preds.flatten()
        targets = targets.flatten()
        
        mask = ~np.isin(targets, list(self.ignore_classes))
        preds = preds[mask]
        targets = targets[mask]
        
        indices = self.num_classes * targets + preds
        m = np.bincount(indices, minlength=self.num_classes ** 2)
        self.confusion_matrix += m.reshape(self.num_classes, self.num_classes)
    
    def compute_iou(self):
        intersection = np.diag(self.confusion_matrix)
        union = (self.confusion_matrix.sum(axis=1) + 
                 self.confusion_matrix.sum(axis=0) - 
                 intersection)
        iou = np.zeros(self.num_classes)
        valid = union > 0
        iou[valid] = intersection[valid] / union[valid]
        return iou
    
    def compute_miou(self):
        ious = self.compute_iou()
        valid_classes = ~np.isin(np.arange(self.num_classes), list(self.ignore_classes))
        return np.mean(ious[valid_classes])
    
    def compute_dice(self):
        intersection = np.diag(self.confusion_matrix)
        sum_rows = self.confusion_matrix.sum(axis=1)
        sum_cols = self.confusion_matrix.sum(axis=0)
        dice = np.zeros(self.num_classes)
        valid = (sum_rows + sum_cols) > 0
        dice[valid] = 2 * intersection[valid] / (sum_rows[valid] + sum_cols[valid])
        return dice

def configure_optimizer(model, lr=6e-5, weight_decay=0.01):
    """SegFormer recommended: lower LR, higher WD"""
    param_groups = [
        {
            'params': [p for n, p in model.named_parameters() 
                      if 'decode_head' in n and p.requires_grad],
            'lr': lr * 2.0,
        },
        {
            'params': [p for n, p in model.named_parameters() 
                      if 'decode_head' not in n and p.requires_grad],
            'lr': lr,
        }
    ]
    optimizer = AdamW(param_groups, weight_decay=weight_decay)
    return optimizer

def get_scheduler(optimizer, num_epochs, steps_per_epoch, warmup_epochs=3):
    total_steps = num_epochs * steps_per_epoch
    warmup_steps = warmup_epochs * steps_per_epoch
    
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

TRAIN_CONFIG = {
    'num_epochs': 20,
    'lr': 2e-4,
    'weight_decay': 0.01,
    'warmup_epochs': 3,
    'grad_clip': 1.0,
    'save_dir': '/content/drive/MyDrive/flood_segmentation/checkpoints',
    'early_stop_patience': 10,
    'accumulation_steps': 4,  
}

os.makedirs(TRAIN_CONFIG['save_dir'], exist_ok=True)
print("Training configuration ready.")

In [ ]:
# Cell 11: Training Loop (FIXED - scheduler per step, bukan per epoch)
from torch.cuda.amp import GradScaler, autocast
import time

class Trainer:
    def __init__(self, model, train_loader, val_loader, loss_fn, optimizer, scheduler, 
                 device, config, use_amp=True):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = loss_fn
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.config = config
        self.use_amp = use_amp and device.type == 'cuda'
        self.scaler = GradScaler() if self.use_amp else None
        
        self.metrics = FloodSegMetrics(NUM_CLASSES, EMPTY_CLASSES)
        self.best_miou = 0.0
        self.patience_counter = 0
        self.history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'lr': []}
        
        self.batch_size = BATCH_SIZE
        self.accum_steps = config.get('accumulation_steps', 1)
        self.steps_per_epoch = len(train_loader)
    
    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        pbar = tqdm(self.train_loader, desc=f'Training E{epoch}')
        
        self.optimizer.zero_grad()
        
        for batch_idx, (images, masks, _) in enumerate(pbar):
            images = images.to(self.device)
            masks = masks.to(self.device)
            
            if self.use_amp:
                with autocast():
                    logits = self.model(images)
                    loss, ce_loss, lovasz_loss = self.loss_fn(logits, masks)
                self.scaler.scale(loss).backward()
                
                if (batch_idx + 1) % self.accum_steps == 0:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['grad_clip'])
                    self.scaler.step(self.optimizer)
                    # FIXED: Scheduler step di sini (per step, bukan per epoch)
                    self.scheduler.step()
                    self.scaler.update()
                    self.optimizer.zero_grad()
            else:
                logits = self.model(images)
                loss, ce_loss, lovasz_loss = self.loss_fn(logits, masks)
                loss.backward()
                
                if (batch_idx + 1) % self.accum_steps == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['grad_clip'])
                    self.optimizer.step()
                    # FIXED: Scheduler step di sini
                    self.scheduler.step()
                    self.optimizer.zero_grad()
            
            total_loss += loss.item()
            
            # Record current learning rate
            current_lr = self.optimizer.param_groups[0]['lr']
            
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'ce': f'{ce_loss.item():.4f}',
                'lov': f'{lovasz_loss.item():.4f}',
                'lr': f'{current_lr:.2e}'
            })
        
        return total_loss / len(self.train_loader)
    
    @torch.no_grad()
    def validate(self):
        self.model.eval()
        self.metrics.reset()
        total_loss = 0
        
        pbar = tqdm(self.val_loader, desc='Validating')
        for images, masks, _ in pbar:
            images = images.to(self.device)
            masks = masks.to(self.device)
            
            logits = self.model(images)
            loss, _, _ = self.loss_fn(logits, masks)
            total_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1)
            self.metrics.update(preds.cpu().numpy(), masks.cpu().numpy())
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(self.val_loader)
        miou = self.metrics.compute_miou()
        ious = self.metrics.compute_iou()
        dice = self.metrics.compute_dice()
        
        # Log per-class
        print("\n" + "="*65)
        print(f"{'Class':<20} {'IoU':>8} {'Dice':>8} {'Pixels':>12}")
        print("-"*65)
        for c in range(NUM_CLASSES):
            if c in EMPTY_CLASSES:
                continue
            name = CLASS_NAMES[c][:18]
            total_pixels = self.metrics.confusion_matrix[c].sum()
            print(f"{name:<20} {ious[c]:>8.3f} {dice[c]:>8.3f} {total_pixels:>12,}")
        print("-"*65)
        print(f"{'mIoU (avg)':<20} {miou:>8.3f}")
        print("="*65)
        
        return avg_loss, miou, ious, dice
    
    def train(self):
        print(f"Starting training for {self.config['num_epochs']} epochs")
        print(f"Device: {self.device}")
        print(f"AMP: {self.use_amp}")
        print(f"Batch size: {self.batch_size}")
        print(f"Grad accumulation: {self.accum_steps}")
        print(f"Steps per epoch: {self.steps_per_epoch}")
        print("="*65)
        
        for epoch in range(1, self.config['num_epochs'] + 1):
            print(f"\nEpoch {epoch}/{self.config['num_epochs']}")
            
            start_time = time.time()
            train_loss = self.train_epoch(epoch)
            
            val_loss, val_miou, ious, dice = self.validate()
            
            # Save history (scheduler sudah step di train_epoch, tidak perlu di sini)
            current_lr = self.optimizer.param_groups[0]['lr']
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_miou'].append(val_miou)
            self.history['lr'].append(current_lr)
            
            epoch_time = time.time() - start_time
            print(f"\nEpoch {epoch} completed in {epoch_time:.1f}s")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | mIoU: {val_miou:.4f} | LR: {current_lr:.2e}")
            
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': self.model.state_dict(),
                'optimizer_state_dict': self.optimizer.state_dict(),
                'scheduler_state_dict': self.scheduler.state_dict(),
                'best_miou': self.best_miou,
                'history': self.history,
                'val_miou': val_miou,
                'ious': ious,
                'dice': dice,
            }
            
            torch.save(checkpoint, os.path.join(self.config['save_dir'], 'last_checkpoint.pth'))
            
            if val_miou > self.best_miou:
                self.best_miou = val_miou
                self.patience_counter = 0
                torch.save(checkpoint, os.path.join(self.config['save_dir'], 'best_model.pth'))
                print(f" New best mIoU: {val_miou:.4f}!")
            else:
                self.patience_counter += 1
                print(f"mIoU did not improve. Patience: {self.patience_counter}/{self.config['early_stop_patience']}")
            
            if self.patience_counter >= self.config['early_stop_patience']:
                print(f"Early stopping triggered after {epoch} epochs")
                break
        
        print(f"\nTraining finished! Best mIoU: {self.best_miou:.4f}")
        
        # Save history ke file JSON agar tidak hilang
        import json
        with open(os.path.join(self.config['save_dir'], 'training_history.json'), 'w') as f:
            # Convert numpy values to Python types
            history_serializable = {k: [float(x) if hasattr(x, 'item') else x for x in v] 
                                   for k, v in self.history.items()}
            json.dump(history_serializable, f, indent=2)
        print(f"History saved to {self.config['save_dir']}/training_history.json")
        
        return self.history

def setup_and_train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    if device.type == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    model = init_model(device)
    
    optimizer = configure_optimizer(model, lr=TRAIN_CONFIG['lr'], weight_decay=TRAIN_CONFIG['weight_decay'])
    scheduler = get_scheduler(
        optimizer, 
        TRAIN_CONFIG['num_epochs'], 
        len(train_loader),
        TRAIN_CONFIG['warmup_epochs']
    )
    
    # Move class weights to device
    loss_fn_device = FloodSegLoss(CLASS_WEIGHTS.to(device), lovasz_weight=0.75)
    
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        loss_fn=loss_fn_device,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        config=TRAIN_CONFIG,
        use_amp=True
    )
    
    history = trainer.train()
    return trainer, history

print("Training pipeline ready. Call setup_and_train() to start training.")

In [ ]:
# Cell 12: Quick Test (FIXED - dengan accumulation_steps eksplisit)
def quick_test():
    print("\n=== QUICK TEST RUN (1 EPOCH) ===")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = init_model(device)
    
    # Use small subset
    small_train = torch.utils.data.Subset(train_dataset, range(32))
    small_val = torch.utils.data.Subset(val_dataset, range(16))
    
    test_train_loader = DataLoader(small_train, batch_size=4, shuffle=True, num_workers=0)
    test_val_loader = DataLoader(small_val, batch_size=4, shuffle=False, num_workers=0)
    
    optimizer = configure_optimizer(model, lr=2e-4, weight_decay=0.01)
    scheduler = get_scheduler(optimizer, 1, len(test_train_loader), 0)
    loss_fn_test = FloodSegLoss(CLASS_WEIGHTS.to(device), lovasz_weight=0.75)
    
    # FIXED: Tambahkan accumulation_steps
    trainer = Trainer(
        model=model,
        train_loader=test_train_loader,
        val_loader=test_val_loader,
        loss_fn=loss_fn_test,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        config={
            'grad_clip': 1.0, 
            'early_stop_patience': 10, 
            'num_epochs': 1, 
            'save_dir': 'test_checkpoints',
            'accumulation_steps': 1  # ← eksplisit
        },
        use_amp=False
    )
    trainer.train()
    print("Quick test completed!")

# Uncomment to test:
# quick_test()